#AI Resume Screening Sys with Tracing
##Objective
LLM pipeline creation, tracing, monitoring, and debugging (using LangChain and LangSmith)

In [1]:
#install req packages
!pip install langchain langchain-core langchain-groq langsmith python-dotenv -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 3.6 MB/s eta 0:00:00


##Configure api keys and enable langsmith tracing

In [16]:
import os

#LangSmith Tracing Configuration
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_PROJECT"]    = "AI-Resume-Screening"
os.environ["LANGCHAIN_API_KEY"]    = "langsmith_API"

#Groq API Key
os.environ["GROQ_API_KEY"]         = "groq_API"

print("[O] Environment variables set")
print(f"Tracing enabled : {os.environ['LANGCHAIN_TRACING_V2']}")
print(f"Project         : {os.environ['LANGCHAIN_PROJECT']}")

[O] Environment variables set
Tracing enabled : true
Project         : AI-Resume-Screening


##LLM utility

In [9]:
from langchain_groq import ChatGroq
def get_llm(temperature: float = 0.0) -> ChatGroq:
    return ChatGroq(
        model="llama-3.3-70b-versatile",
        temperature=temperature
    )

#test
llm_test = get_llm()
response = llm_test.invoke("Say 'LLM Ready' in exactly 2 words.")
print(f"LLM check: {response.content.strip()}")

LLM check: LLM Ready


##Sample data

In [17]:
# ─── Job Description ────────────────────────────────────────────────────────
JOB_DESCRIPTION = """
Position: Machine Learning Engineer
Company: NovaData Systems

Required Skills:
- Python (4+ years)
- Machine Learning (scikit-learn, CatBoost, XGBoost)
- Deep Learning (PyTorch or TensorFlow)
- SQL and data processing (pandas, numpy)
- Statistical analysis and experimentation
- Data visualization (Matplotlib, Plotly, Seaborn)
- Cloud platforms (AWS, Azure, or GCP)
- MLOps tools (Docker, MLflow, CI/CD pipelines)
- Strong problem-solving and communication skills

Experience Required: 3–6 years in ML/Data Science roles
Education: Bachelor's or Master's in Computer Science, Data Science, or related field
"""

# ─── Strong Candidate ────────────────────────────────────────────────────────
RESUME_STRONG = """
Name: Candidate A
Experience: 5 years as ML Engineer at Swiggy Data Labs
Education: M.S. in Data Science, IISc Bangalore

Skills:
- Python (7 years), SQL, Scala
- Machine Learning: scikit-learn, XGBoost, CatBoost, LightGBM
- Deep Learning: PyTorch, TensorFlow
- Data tools: pandas, numpy, Spark, Hive
- Cloud: AWS (Lambda, S3, SageMaker), Azure ML
- MLOps: Docker, Kubernetes, MLflow, Airflow, CI/CD
- Visualization: Plotly, Seaborn, Tableau
- Statistics: hypothesis testing, A/B testing, time-series analysis

Projects:
- Designed large-scale recommendation engine for 8M+ users (XGBoost + Spark)
- Built deep learning-based delivery time prediction system (PyTorch + AWS)
- Automated ML pipeline reducing model deployment time by 60%

Certifications: AWS Certified Solutions Architect, Azure AI Engineer Associate
"""

# ─── Average Candidate ───────────────────────────────────────────────────────
RESUME_AVERAGE = """
Name: Candidate B
Experience: 3 years as Data Analyst at TCS
Education: B.Tech in Computer Science, SRM University

Skills:
- Python (3 years), SQL
- Machine Learning: scikit-learn (basic classification/regression)
- Data tools: pandas, numpy, Excel
- Visualization: Power BI, Matplotlib
- Basic cloud exposure (AWS EC2, S3)

Projects:
- Built sales forecasting model using linear regression
- Created dashboards for KPI tracking in Power BI
- Performed data cleaning and preprocessing for retail datasets

Certifications: Coursera Machine Learning Specialization
"""

# ─── Weak Candidate ──────────────────────────────────────────────────────────
RESUME_WEAK = """
Name: Candidate C
Experience: 8 months internship as IT Support Assistant
Education: B.A. in Economics

Skills:
- Basic Python (beginner)
- MS Excel, Word, PowerPoint
- Basic SQL (intro level)
- Familiar with HTML and CSS

Projects:
- Created Excel reports for internal team usage
- Built a basic to-do list app using Python
- Assisted in maintaining company databases

Certifications: Basic Python Bootcamp (Udemy)
"""

print("[O] Resume data loaded:")
print(f"Strong  : Candidate A (5 yrs, M.S.)")
print(f"Average : Candidate B (3 yrs, B.Tech)")
print(f"Weak    : Candidate C (8 months intern, B.A.)")

[O] Resume data loaded:
Strong  : Candidate A (5 yrs, M.S.)
Average : Candidate B (3 yrs, B.Tech)
Weak    : Candidate C (8 months intern, B.A.)


##Prompt templates

In [19]:
from langchain_core.prompts import PromptTemplate

extraction_prompt = PromptTemplate(
    input_variables=["resume"],
    template="""
You are an accurate resume information extractor.

Strict rules:
- Use ONLY the information explicitly present in the resume.
- Do NOT infer, guess, or add missing details.
- If a field is not available, return null (or an empty list where applicable).
- Keep responses concise and structured.

Resume Text:
{resume}

Extract and return EXACTLY this JSON format:
{{
  "name": "string or null",
  "years_experience": number or null,
  "education": "string or null",
  "skills": ["..."],
  "tools": ["..."],
  "key_projects": ["..."]
}}

Constraints:
- "years_experience" must be a number (no text like "3 years").
- "skills" = core competencies (e.g., Machine Learning, Statistics).
- "tools" = technologies/libraries/platforms (e.g., Python, AWS, TensorFlow).
- "key_projects" = short one-line summaries extracted from the resume.

Output rules:
- Return ONLY valid JSON.
- No markdown, no explanations, no extra text.
"""
)

print("[O] extraction_prompt loaded")

[O] extraction_prompt loaded


In [18]:
matching_prompt = PromptTemplate(
    input_variables=["extracted_profile", "job_description"],
    template="""
You are an experienced recruiter conducting a structured skill-gap and fit analysis.

Inputs:
- Job Description
- Candidate Profile (already extracted)

Job Description:
{job_description}

Candidate Profile:
{extracted_profile}

Evaluation Rules:
- Compare ONLY the candidate's listed skills, tools, experience, and education against the job requirements.
- Do NOT assume or infer any missing skills or experience.
- A skill is "matched" ONLY if it clearly appears in both job requirements and candidate profile.
- Be strict and consistent in evaluation.

Scoring Guidelines:
- experience_match:
  - "strong" = meets or exceeds required years and relevant domain experience
  - "partial" = slightly below or lacks depth in domain
  - "weak" = clearly below requirement or irrelevant experience

- education_match:
  - "strong" = directly matches or exceeds required field/degree
  - "partial" = related but not exact
  - "weak" = unrelated or missing

- overall_match:
  - "strong" = majority of critical skills matched + strong experience
  - "partial" = moderate skill match and/or experience gaps
  - "weak" = significant skill and experience gaps

Output Format (STRICT):
{{
  "matched_skills": ["..."],
  "missing_skills": ["..."],
  "experience_match": "strong | partial | weak",
  "education_match": "strong | partial | weak",
  "overall_match": "strong | partial | weak"
}}

Output Rules:
- Return ONLY valid JSON.
- No markdown, no explanations, no extra text.
- Ensure all arrays contain unique, concise entries.
"""
)

print("[O] matching_prompt loaded")

[O] matching_prompt loaded


In [20]:
scoring_prompt = PromptTemplate(
    input_variables=["extracted_profile", "match_analysis", "job_description"],
    template="""
You are a deterministic AI hiring evaluator assigning a numerical fit score.

Inputs:
- Job Description
- Candidate Profile (structured)
- Match Analysis (skills gap + qualitative evaluation)

Job Description:
{job_description}

Candidate Profile:
{extracted_profile}

Match Analysis:
{match_analysis}

Scoring Framework (Total = 100):

1. Skills Match (0–40)
- Score proportionally based on required skills present in the candidate profile.
- Core/critical skills should weigh more than secondary ones.

2. Experience Match (0–25)
- "strong" → 20–25
- "partial" → 10–19
- "weak" → 0–9
- Base this on years + domain relevance.

3. Education Match (0–15)
- "strong" → 12–15
- "partial" → 6–11
- "weak" → 0–5

4. Tools & MLOps (0–20)
- Evaluate presence of cloud platforms, MLOps tools, and production-level tooling.
- Score higher for breadth + real-world usage.

Strict Rules:
- Use ONLY explicitly stated information.
- Do NOT infer or assume missing skills/tools/experience.
- Keep scoring consistent with the match analysis provided.
- Ensure total_score = sum of all component scores.

Grading Logic:
- A: 85–100 → Strongly Recommend
- B: 70–84 → Recommend
- C: 50–69 → Consider
- D: <50 → Reject

Output Format (STRICT JSON ONLY):
{{
  "skills_score": number,
  "experience_score": number,
  "education_score": number,
  "tools_score": number,
  "total_score": number,
  "grade": "A | B | C | D",
  "recommendation": "Strongly Recommend | Recommend | Consider | Reject",
  "explanation": "2-3 concise sentences referencing specific evidence from the candidate profile"
}}

Output Rules:
- Return ONLY valid JSON.
- No markdown, no commentary, no extra text.
- All numeric fields must be integers.
"""
)

print("[O] scoring_prompt loaded")

[O] scoring_prompt loaded


##LangChain LCEL Chains

In [21]:
# Low temperature = deterministic extraction (no hallucination)
extraction_llm = get_llm(temperature=0.0)
extraction_chain = extraction_prompt | extraction_llm

# Low temperature = objective comparison
matching_llm = get_llm(temperature=0.0)
matching_chain = matching_prompt | matching_llm

# Slightly higher temperature for nuanced explanation
scoring_llm = get_llm(temperature=0.1)
scoring_chain = scoring_prompt | scoring_llm

print("[O] All LCEL chains created:")
print("extraction_chain  = extraction_prompt | llm(temp=0.0)")
print("matching_chain    = matching_prompt   | llm(temp=0.0)")
print("scoring_chain     = scoring_prompt    | llm(temp=0.1)")

[O] All LCEL chains created:
extraction_chain  = extraction_prompt | llm(temp=0.0)
matching_chain    = matching_prompt   | llm(temp=0.0)
scoring_chain     = scoring_prompt    | llm(temp=0.1)


##Main Pipeline

In [22]:
import json

def safe_parse_json(text: str) -> dict:

    # Strip markdown fences if present
    cleaned = text.strip()
    if cleaned.startswith("```"):
        lines = cleaned.split("\n")
        cleaned = "\n".join(lines[1:-1])  # remove first and last fence lines
    try:
        return json.loads(cleaned)
    except json.JSONDecodeError as e:
        return {"parse_error": str(e), "raw": text[:300]}


def screen_resume(resume: str, job_description: str, candidate_type: str) -> dict:

    print(f"\n{'='*60}")
    print(f"Screening: {candidate_type} Candidate")
    print(f"{'='*60}")

    #Step 1: Skill Extraction
    print("\n[1] : Extracting skills and profile...")
    extraction_result = extraction_chain.invoke({"resume": resume})
    extracted_profile = extraction_result.content.strip()
    profile_data = safe_parse_json(extracted_profile)
    print(f"   Name        : {profile_data.get('name', 'N/A')}")
    print(f"   Experience  : {profile_data.get('years_experience', 'N/A')} years")
    print(f"   Education   : {profile_data.get('education', 'N/A')}")
    print(f"   Skills count: {len(profile_data.get('skills', []))}")

    #Step 2: Matching Logic
    print("\n[2] : Matching profile against job requirements...")
    matching_result = matching_chain.invoke({
        "extracted_profile": extracted_profile,
        "job_description": job_description
    })
    match_analysis = matching_result.content.strip()
    match_data = safe_parse_json(match_analysis)
    print(f"   Matched skills : {len(match_data.get('matched_skills', []))}")
    print(f"   Missing skills : {len(match_data.get('missing_skills', []))}")
    print(f"   Overall match  : {match_data.get('overall_match', 'N/A')}")

    #Step 3: Scoring + Explanation
    print("\n[3] : Calculating fit score and explanation...")
    scoring_result = scoring_chain.invoke({
        "extracted_profile": extracted_profile,
        "match_analysis": match_analysis,
        "job_description": job_description
    })
    score_text = scoring_result.content.strip()
    score_data = safe_parse_json(score_text)
    print(f"Total Score    : {score_data.get('total_score', 'N/A')}/100")
    print(f"Grade          : {score_data.get('grade', 'N/A')}")
    print(f"Recommendation : {score_data.get('recommendation', 'N/A')}")

    return {
        "candidate_type" : candidate_type,
        "profile"        : profile_data,
        "match_analysis" : match_data,
        "scoring"        : score_data
    }

print("[O] screen_resume() pipeline function defined")

[O] screen_resume() pipeline function defined


##Run 3 screening sessions (strong, average, weak)

In [23]:
#Run 1: Strong Candidate
result_strong = screen_resume(
    resume=RESUME_STRONG,
    job_description=JOB_DESCRIPTION,
    candidate_type="STRONG"
)


Screening: STRONG Candidate

[1] : Extracting skills and profile...
   Name        : Candidate A
   Experience  : 5 years
   Education   : M.S. in Data Science, IISc Bangalore
   Skills count: 3

[2] : Matching profile against job requirements...
   Matched skills : 17
   Missing skills : 5
   Overall match  : strong

[3] : Calculating fit score and explanation...
Total Score    : 86/100
Grade          : A
Recommendation : Strongly Recommend


In [24]:
#Run 2: Average Candidate
result_average = screen_resume(
    resume=RESUME_AVERAGE,
    job_description=JOB_DESCRIPTION,
    candidate_type="AVERAGE"
)


Screening: AVERAGE Candidate

[1] : Extracting skills and profile...
   Name        : Candidate B
   Experience  : 3 years
   Education   : B.Tech in Computer Science, SRM University
   Skills count: 2

[2] : Matching profile against job requirements...
   Matched skills : 7
   Missing skills : 13
   Overall match  : partial

[3] : Calculating fit score and explanation...
Total Score    : 50/100
Grade          : C
Recommendation : Consider


In [25]:
#Run 3: Weak Candidate
result_weak = screen_resume(
    resume=RESUME_WEAK,
    job_description=JOB_DESCRIPTION,
    candidate_type="WEAK"
)


Screening: WEAK Candidate

[1] : Extracting skills and profile...
   Name        : Candidate C
   Experience  : 0.67 years
   Education   : B.A. in Economics
   Skills count: 1

[2] : Matching profile against job requirements...
   Matched skills : 2
   Missing skills : 6
   Overall match  : weak

[3] : Calculating fit score and explanation...
Total Score    : 8/100
Grade          : D
Recommendation : Reject


##Results summary and comparison

In [26]:
print("\n" + "="*70)
print("RESUME SCREENING RESULTS SUMMARY")
print("="*70)
print(f"{'Candidate':<12} {'Name':<18} {'Score':>7} {'Grade':>7} {'Recommendation':<25}")
print("-"*70)

for result in [result_strong, result_average, result_weak]:
    name     = result["profile"].get("name", "N/A")
    score    = result["scoring"].get("total_score", "N/A")
    grade    = result["scoring"].get("grade", "N/A")
    rec      = result["scoring"].get("recommendation", "N/A")
    ctype    = result["candidate_type"]
    print(f"{ctype:<12} {name:<18} {str(score):>7} {str(grade):>7} {rec:<25}")

print("="*70)

print("\n📋 Score Breakdown:")
print(f"{'Candidate':<12} {'Skills':>8} {'Exp':>6} {'Edu':>6} {'Tools':>7} {'Total':>7}")
print("-"*50)
for result in [result_strong, result_average, result_weak]:
    sc    = result["scoring"]
    ctype = result["candidate_type"]
    print(f"{ctype:<12} {str(sc.get('skills_score','?')):>8} "
          f"{str(sc.get('experience_score','?')):>6} "
          f"{str(sc.get('education_score','?')):>6} "
          f"{str(sc.get('tools_score','?')):>7} "
          f"{str(sc.get('total_score','?')):>7}")


RESUME SCREENING RESULTS SUMMARY
Candidate    Name                 Score   Grade Recommendation           
----------------------------------------------------------------------
STRONG       Candidate A             86       A Strongly Recommend       
AVERAGE      Candidate B             50       C Consider                 
WEAK         Candidate C              8       D Reject                   

📋 Score Breakdown:
Candidate      Skills    Exp    Edu   Tools   Total
--------------------------------------------------
STRONG             32     22     14      18      86
AVERAGE            16     15     12       7      50
WEAK                8      0      0       0       8


##Detailed explaination

In [27]:
for result in [result_strong, result_average, result_weak]:
    name        = result["profile"].get("name", "N/A")
    ctype       = result["candidate_type"]
    explanation = result["scoring"].get("explanation", "No explanation generated.")
    matched     = result["match_analysis"].get("matched_skills", [])
    missing     = result["match_analysis"].get("missing_skills", [])

    print(f"\n{'─'*60}")
    print(f"{name} ({ctype} Candidate)")
    print(f"{'─'*60}")
    print(f"Explanation : {explanation}")
    print(f"Matched     : {', '.join(matched[:5])}{'...' if len(matched) > 5 else ''}")
    print(f"Missing     : {', '.join(missing[:5])}{'...' if len(missing) > 5 else ''}")


────────────────────────────────────────────────────────────
Candidate A (STRONG Candidate)
────────────────────────────────────────────────────────────
Explanation : Candidate A has a strong match in skills, including Machine Learning, Deep Learning, and Python, with relevant experience in ML/Data Science roles, and a Master's degree in Data Science from IISc Bangalore, with notable projects such as a large-scale recommendation engine and a deep learning-based delivery time prediction system.
Matched     : Machine Learning, Deep Learning, Python, SQL, scikit-learn...
Missing     : Statistical analysis and experimentation, Data visualization (Matplotlib), Azure, GCP, MLOps tools (beyond Docker, MLflow, CI/CD pipelines)

────────────────────────────────────────────────────────────
Candidate B (AVERAGE Candidate)
────────────────────────────────────────────────────────────
Explanation : Candidate B has 3 years of experience and a strong education match, but lacks critical skills like De

##LangSmith debugging

In [29]:
RESUME_EDGE_CASE = """
Name: Unknown Person
I am a quick learner and hardworking individual.
I know computers and internet.
I want to work in your company.
"""

print("Running edge case to demonstrate LangSmith debugging...")
print("Input: Vague resume with no specific skills or experience")
print()

# Extract only — to show incorrect/minimal output captured in LangSmith
edge_extraction = extraction_chain.invoke({"resume": RESUME_EDGE_CASE})
edge_data = safe_parse_json(edge_extraction.content)

print("Extraction result (edge case):")
print(json.dumps(edge_data, indent=2))

print()
print("LangSmith Debugging Insight:")
print("- The model correctly returns 0 years_experience and empty skills list")
print("- This is the CORRECT behavior — no hallucination of skills")
print("- LangSmith trace shows exactly what prompt was sent and what was returned")
print("- Useful for identifying prompts that produce unexpected outputs")
print("View trace at: https://smith.langchain.com/ → Project: AI-Resume-Screening")

Running edge case to demonstrate LangSmith debugging...
Input: Vague resume with no specific skills or experience

Extraction result (edge case):
{
  "name": "Unknown Person",
  "years_experience": null,
  "education": null,
  "skills": [
    "quick learning",
    "hard work"
  ],
  "tools": [
    "computers",
    "internet"
  ],
  "key_projects": []
}

LangSmith Debugging Insight:
- The model correctly returns 0 years_experience and empty skills list
- This is the CORRECT behavior — no hallucination of skills
- LangSmith trace shows exactly what prompt was sent and what was returned
- Useful for identifying prompts that produce unexpected outputs
View trace at: https://smith.langchain.com/ → Project: AI-Resume-Screening
